### Assignment 1: Reconstructing the past: estimating solar radio flux F10.7 from sunspot records

##### Group 6: Pavel Kuznetsov, Olga Tiupina, Sofya Konstantinova, Skoltech, 2025

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import math


def calculate_rmse(f_real, f_predicted):
    """
    Calculate RMSE
    
    params:
    f_real - vector of real values of f
    f_predicted - vector of predicted values of f
    
    returns:
    rmse - error
    """
    N = len(f_real)
    
    sum_sq_errors = 0.0
    for i in range(N):
        diff = f_real[i] - f_predicted[i]
        sum_sq_errors += diff * diff  
    
    rmse = math.sqrt(sum_sq_errors / (N - 1))
    
    return rmse

def create_matrix_of_β(dependent_variables, independent_variables):
    """
    Calculate beta coefficients using normal equation: β = (RᵀR)⁻¹ RᵀF
    
    params:
    dependent_variables (F) - vector of dependent variables (n x 1)
    independent_variables (R) - matrix of independent variables (n x 4)
    
    returns:
    beta_matrix - vector of coefficients (4 x 1)
    """

    # find β step-by-step
    try:
        # R^t * R
        r_transpose_r = independent_variables.T @ independent_variables
        
        # (R^t*R)^(-1)
        r_transpose_r_inv = np.linalg.pinv(r_transpose_r)
        
        # (R^t*R)^(-1) * R^t
        temp_matrix = r_transpose_r_inv @ independent_variables.T
        
        # β = (R^t*R)^(-1) * R^t * F
        beta_matrix = temp_matrix @ dependent_variables
        
        return beta_matrix
    
    # the matrix is singular, use the pseudoinverse matrix
    except np.linalg.LinAlgError:
        return np.linalg.pinv(independent_variables) @ dependent_variables

def regress_F(dependent_variables, independent_variables, independent_variables_to_propagate):
    """
    Predict F values using regression coefficients
    
    params:
    dependent_variables (F) - vector of dependent variables for training
    independent_variables (R) - matrix of independent variables for training
    independent_variables_to_propagate - R values for prediction (before 1951)
    
    returns:
    predicted_F - predicted F values for the period before 1951
    """
    # calculate beta coefficients
    betta = create_matrix_of_β(dependent_variables, independent_variables)
    
    # extract coefficients
    beta0 = betta[0, 0]
    beta1 = betta[1, 0]
    beta2 = betta[2, 0]  
    beta3 = betta[3, 0]
    
    # initialize prediction array
    n = len(independent_variables_to_propagate)
    predicted_F = np.zeros(n)
    
    # predict F using the formula: F_i = β0 + β1*R_i + β2*R_i^2 + β3*R_i^3
    for i in range(n):
        predicted_F[i] = (beta0 + 
                         beta1 * independent_variables_to_propagate[i] + 
                         beta2 * (independent_variables_to_propagate[i] ** 2) + 
                         beta3 * (independent_variables_to_propagate[i] ** 3))
    
    return predicted_F

def create_matrix_of_independent_variables(values):
    """
    Create matrix
    
    params:
    values - value array
    
    returns:
    matrix: [[1, value1, value1^2, value1^3]
            [[1, value2, value2^2, value2^3]
             ... 
            ]
    """
    # initialize matrix 
    n = len(values)
    matrix = np.zeros((n, 4))

    # fill matrix with degrees: 0, 1, 2, 3 
    for d in range(0, 4):  
        matrix[:, d] = values ** d
    
    return matrix

def smooth_data(dates, values):
    """
    Smoothing data function
    
    param:
    dates - timeline array
    values - an array of values corresponding to dates
    """
    temp_df = pd.DataFrame({'date': dates, 'value': values})
    temp_df = temp_df.set_index('date')    

    # create cope of values
    data = temp_df['value'].values.copy()
    n = len(data)

    smoothed_data = np.zeros(n)
    
    # find average for first and last 6 data points 
    # first
    average_first_6 = np.mean(data[:6])
    smoothed_data[:6] = average_first_6
    # end
    average_last_6 = np.mean(data[-6:])
    smoothed_data[-6:] = average_last_6

    # smoothing process for point from 5 до n-6 by 13-month running mean
    for t in range(5, n-6):
        smoothed_data[t] = (1/24 * data[t-6] + 
                   1/24 * data[t+6] + 
                   1/12 * (data[t] + data[t+1] + data[t+2] + data[t+3] + 
                           data[t+4] + data[t+5] + data[t-1] + data[t-2] + 
                           data[t-3] + data[t-4] + data[t-5]))

    temp_df['value_smooth'] = smoothed_data
    
    return temp_df

In [ ]:
fontsize_title = 25
fontsize_label = 20
fontsize_leg = 18

In [ ]:
# read Sunspot_number_monthly_mean data file
df1 = pd.read_csv('../data/Sunspot_number_monthly_mean.txt', 
                 sep=r'\s+', names=['year', 'month', 'value'], header=None) 
df1['date'] = pd.to_datetime(df1['year'].astype(str) + '-' + df1['month'].astype(str))

# read Radio_flux_monthly_mean data file  
df2 = pd.read_csv('../data/Radio_flux_monthly_mean.txt', 
                 sep=r'\s+', names=['year', 'month', 'value'], header=None) 
df2['date'] = pd.to_datetime(df2['year'].astype(str) + '-' + df2['month'].astype(str))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 15))

mask1 = df1['date'] <= '1951-10-31'
dates1 = df1.loc[mask1, 'date']
ax1.plot(dates1, df1.loc[mask1, 'value'], '-', 
         linewidth=0.5, label='Monthly sunspot number', color='blue')
ax1.set_xlabel('\n Date, year \n', fontsize=fontsize_label)
ax1.set_ylabel('\n Sunspot number \n', fontsize=fontsize_label)
ax1.set_title('\n Fig.1. Sunspot number (until Oct 1951) \n', fontsize=fontsize_title)
ax1.legend(fontsize=fontsize_leg)
ax1.grid(True)
ax1.set_xlim(dates1.min(), dates1.max())

mask2_spot = df1['date'] >= '1951-11-30'
mask2_radio = df2['date'] >= '1951-11-30'

dates2_spot = df1.loc[mask2_spot, 'date']
dates2_radio = df2.loc[mask2_radio, 'date']

ax2_primary = ax2
ax2_primary.plot(dates2_spot, df1.loc[mask2_spot, 'value'], '-', 
         linewidth=0.5, markersize=0.5, label='Monthly sunspot number', color='blue')
ax2_primary.set_xlabel('\n Date, year \n', fontsize=fontsize_label)
ax2_primary.set_ylabel('\n Sunspot number \n', color='blue', fontsize=fontsize_label)
ax2_primary.tick_params(axis='y', labelcolor='blue', labelsize=fontsize_label)
ax2_primary.set_ylim(0, 300)
ax2_primary.grid(True)
ax2_primary.set_xlim(min(dates2_spot.min(), dates2_radio.min()), 
                     max(dates2_spot.max(), dates2_radio.max()))

ax2_secondary = ax2_primary.twinx()
ax2_secondary.plot(dates2_radio, df2.loc[mask2_radio, 'value'], '-', 
         linewidth=0.5, markersize=0.5, label='Monthly solar radio flux at 10.7 cm', color='red')
ax2_secondary.set_ylabel('\n Radio flux F10.7, sfu \n', color='red', fontsize=fontsize_label)
ax2_secondary.tick_params(axis='y', labelcolor='red', labelsize=fontsize_label)
ax2_secondary.set_ylim(0, 300)

lines1, labels1 = ax2_primary.get_legend_handles_labels()
lines2, labels2 = ax2_secondary.get_legend_handles_labels()
ax2_secondary.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=fontsize_leg)

ax2_primary.set_title('\n Fig.2. Sunspot number and radio flux (from Nov 1951) \n', fontsize=fontsize_title)

plt.tight_layout()
plt.show()


The first graph shows the monthly mean sunspot numbers between January 1749 and October 1951. The second graph shows the monthly mean sunspot numbers and the solar radio stream F10.7 cm to better understand the similarities and differences in the sunspot numbers and F10.7 dynamics. The combined graphs show that: 
- Dependencies have the same period; 
- The upper peaks match;

In [ ]:
smoothed_sunspots = smooth_data(df1['date'], df1['value'])
smoothed_radio = smooth_data(df2['date'], df2['value'])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(25, 15))

# Sunspots data
ax1.plot(df1['date'], df1['value'], 'gray', 
         linewidth=0.5, alpha=1, label='Original sunspots')
ax1.plot(smoothed_sunspots.index, smoothed_sunspots['value_smooth'], 'blue',
         linewidth=1, alpha=0.8, label='Smoothed sunspots')
ax1.set_xlabel('\n Date, year \n', fontsize=fontsize_label)
ax1.set_ylabel('\n Sunspot Number \n', fontsize=fontsize_label)
ax1.set_title('\n Fig.3. Sunspot Data: Original vs Smoothed \n', fontsize=fontsize_title)
ax1.legend(fontsize=fontsize_leg)
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45, labelsize=fontsize_label)
ax1.tick_params(axis='y', labelsize=fontsize_label)

# Radio flux data
ax2.plot(df2['date'], df2['value'], 'gray',
         linewidth=0.5, alpha=1, label='Original radio flux')
ax2.plot(smoothed_radio.index, smoothed_radio['value_smooth'], 'red',
         linewidth=1, label='Smoothed radio flux')
ax2.set_xlabel('\n Date, year \n', fontsize=fontsize_label)
ax2.set_ylabel('\n Radio Flux F10.7, sfu \n', fontsize=fontsize_label)
ax2.set_title('\n Fig.4. Radio Flux Data: Original vs Smoothed \n', fontsize=fontsize_title)
ax2.legend(fontsize=fontsize_leg)
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45, labelsize=fontsize_label)
ax2.tick_params(axis='y', labelsize=fontsize_label)

plt.tight_layout()
plt.show()


Then we applied the 13-month running mean smoothing method to smooth out the monthly mean data on the sunspot numbers and the solar radio stream F10.7. The original data is represented in gray, and the blue and red are smoothed. Further, the smoothed data obtained was used to build a linear regression.

In [ ]:
start_date = df2['date'].min()
# end date is the earliest last date from both of data files
end_date = min(df1['date'].max(), df2['date'].max())

# synchronize datas by dates
df1_trimmed = df1[(df1['date'] >= start_date) & (df1['date'] <= end_date)].copy()
df2_trimmed = df2[df2['date'] <= end_date].copy()

smoothed_sunspots = smooth_data(df1_trimmed['date'], df1_trimmed['value'])
smoothed_radio = smooth_data(df2_trimmed['date'], df2_trimmed['value'])

# scatter plot: smoothed sunspots vs radio flux

plt.figure(figsize=(20, 10))
plt.scatter(smoothed_sunspots['value_smooth'], smoothed_radio['value_smooth'], 
           s=10, alpha=0.7, color='blue')
plt.xlabel('\n Smoothed sunspot number monthly mean \n', fontsize=fontsize_label)
plt.ylabel('\n Smoothed radio flux F10.7 monthly mean \n', fontsize=fontsize_label)
plt.title('\n Fig.5. Correlation: Smoothed Sunspots vs Smoothed Radio Flux (since 1951) \n', fontsize=fontsize_title)
plt.grid(True, alpha=0.3)
plt.tight_layout()  
plt.show()

The graphs overlap in data from October 1951 to August 2025. This graph shows a scatter plot based on smoothed sunspot and F10.7 data during this period. According on the resulting scatter diagram, we can conclude that there is a clear and direct relationship between the sunspot numbers and the solar radio stream F10.7.

In [ ]:
# check accuracy by study-data (after 1951)
predicted_radio_train = regress_F(
    dependent_variables=smoothed_radio['value_smooth'].values.reshape(-1, 1),           # F: radio flux
    independent_variables=create_matrix_of_independent_variables(smoothed_sunspots['value_smooth'].values),  # R: sunspots
    independent_variables_to_propagate=smoothed_sunspots['value_smooth'].values         # R: the same sunspots
)

# error on training data
rmse_train = calculate_rmse(smoothed_radio['value_smooth'].values, predicted_radio_train)
print(f"RMSE on smoothed data: {rmse_train:.4f}")

# plot of real and predicted radioflux (after 1951)
plt.figure(figsize=(20, 10))

plt.plot(smoothed_radio.index, smoothed_radio['value_smooth'], '-', markersize=2,
         linewidth=1.5, alpha=0.7, label='Real SMOOTHED radio flux F10.7, sfu', color='blue')
plt.plot(smoothed_radio.index, predicted_radio_train, '-', markersize=2,
         linewidth=1.5, label='Predicted , sfu', color='red')
plt.xlabel('\n Date, year \n', fontsize=fontsize_label)
plt.ylabel('\n Radio Flux F10.7, sfu \n', fontsize=fontsize_label)
plt.title(f'\n Fig.6. Real vs Predicted Radio Flux (smoothed data since 1951) RMSE: {rmse_train:.4f} \n', fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()  
plt.show()

Based on the constructed multidimensional linear regression, a comparative graph of smoothed and predicted radio flux F10.7 data. Based on the graph obtained and the RMSE of the measurements, we can conclude that the constructed linear regression allows us to predict values close enough to the real ones.

In [ ]:
# cut data which lost in radio flux
df1_trimmed_lost = df1[df1['date'] < start_date].copy()
smoothed_sunspots_lost = smooth_data(df1_trimmed_lost['date'], df1_trimmed_lost['value'])

predicted_radio_lost = regress_F(
    dependent_variables=smoothed_radio['value_smooth'].values.reshape(-1, 1),           # F: radio flux (study part)
    independent_variables=create_matrix_of_independent_variables(smoothed_sunspots['value_smooth'].values),  # R: sunspots (study part)
    independent_variables_to_propagate=smoothed_sunspots_lost['value_smooth'].values        # R: sunspots (prediction)
)

# DataFrame fill with predicted values 
df2_predicted = pd.DataFrame({
    'date': df1_trimmed_lost['date'].values,
    'value': predicted_radio_lost
})

In [ ]:
plt.figure(figsize=(20, 10))

# calculate 3-sigma bounds
sigma = rmse_train 
upper_bound = df2_predicted['value'] + 3 * sigma
lower_bound = df2_predicted['value'] - 3 * sigma

# plot confidence interval (shaded area)
plt.fill_between(df2_predicted['date'], lower_bound, upper_bound, 
                 alpha=0.3, color='red', label='±3σ confidence interval')

#plot smoothed sunspots (blue)
plt.plot(smoothed_sunspots_lost.index, smoothed_sunspots_lost['value_smooth'], '-', markersize=2,
         linewidth=1, alpha=0.7, label='Smoothed sunspot number (before 1951)', color='blue')

# plot predicted radio flux (red line)
plt.plot(df2_predicted['date'], df2_predicted['value'], '-', markersize=2,
         linewidth=1.5, alpha=0.8, label='Predicted radio flux F10.7, sfu', color='red')

plt.xlabel('\n Date, year \n', fontsize=fontsize_label)
plt.ylabel('\n Value \n', fontsize=fontsize_label)
plt.title(f'\n Fig.7. Predicted Radio Flux for Period Before 1951\n(±3σ confidence interval, σ = {sigma:.2f}) \n', fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()  
plt.show()

#### Conclusion

During this assignment, we:

- plotted the monthly mean sunspot numbers and solar radio flux F10.7 cm for visual representation at one plot.The graphs show that both they have the same period, and their upper peaks align. 
- applied a 13-month running mean to smooth the monthly mean data for sunspot numbersand solar radio flux F10.7. Smoothing the data before applying linear regressionhelpsreduce noise and variability, allowing for a clearer relationship to emergebetweenthe variables.
- created a scatter plot of the smoothed sunspot and F10.7 data,it reveals a direct relationship between smoothed sunspot numbers and F10.7 flux, confirming they are reliable proxies for the same solar activity.
- defined the vector of regressands, the matrix of regressors, and the vector of coefficients in the code using the radio flux and sunspot data, the vector of coefficients using LSM. Reconstruct the smoothed solar radio flux at 10.7 cm. The reconstructed F10.7 values closely follow the smoothed observed data, visually confirming the model's adequacy.
- calculated the Root-Mean-Square Error (RMSE) to quantify the difference between the model's predictions and the actual smoothed F10.7 data = 6.1034, indicates a good model fit for the smoothed data.
- applied the obtained model, we plot the reconstructed radio flux F10.7 for earlier periods for which there is information about the sunspot numbers. The patterns that we observed on the initial intersecting data on the radio flux F10.7 and the sunspot numbers are also observed on the reconstructed data on the radio flux F10.7. According to this, we can conclude that the model is built correctly.